# 02b — Multi-CW Optimizer Results Dashboard

Loads JSON outputs from `02_multicw_pairwise_outputs/` and summarizes whether deterministic pairwise scans recover pulsar distances.

Primary metrics:

- `frac_within_half_mode`: fraction of selected pulsars recovered to within half the minimum CW mode spacing.
- `final_lnL - truth_lnL`: optimizer final likelihood relative to truth.
- `median_abs_sigma`: median distance error in EM-prior sigma units.
- convergence curves across sweeps and chains.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTDIR = Path("02_multicw_pairwise_outputs")
files = sorted(OUTDIR.glob("run_*.json"))
print(f"found {len(files)} result files")
files[-5:] if files else []


In [ ]:
def load_run(path):
    data = json.loads(path.read_text())
    rows = []
    hist_rows = []
    cfg = data.get("config", {})
    for res in data.get("results", []):
        row = {
            "file": path.name,
            "chain_id": res["chain_id"],
            "init_kind": res["init_kind"],
            "final_lnL": res["final_lnL"],
            "truth_lnL": data["truth_lnL"],
            "delta_lnL_truth": res["final_lnL"] - data["truth_lnL"],
            **{f"cfg_{k}": v for k, v in cfg.items()},
            **res["final_score"],
        }
        rows.append(row)
        for h in res.get("history", []):
            hist_rows.append({
                "file": path.name,
                "chain_id": res["chain_id"],
                "init_kind": res["init_kind"],
                **{f"cfg_{k}": v for k, v in cfg.items()},
                **h,
            })
    return rows, hist_rows

rows, hist = [], []
for f in files:
    r, h = load_run(f)
    rows.extend(r); hist.extend(h)
summary = pd.DataFrame(rows)
history = pd.DataFrame(hist)
if len(summary):
    summary = summary[np.isfinite(summary["truth_lnL"]) & np.isfinite(summary["final_lnL"])]
if len(history):
    history = history[np.isfinite(history["lnL"])]
print(summary.shape, history.shape)
summary.tail()



In [ ]:
if len(summary):
    cols = [
        "file", "chain_id", "init_kind", "cfg_data_mode", "cfg_n_cw", "cfg_log10_h",
        "cfg_n_psr", "cfg_gwb_log10_a", "delta_lnL_truth", "frac_within_half_mode",
        "median_abs_sigma", "median_abs_modes",
    ]
    cols = [c for c in cols if c in summary.columns]
    display(summary[cols].tail(20))
else:
    print("No results yet. Run 02a first.")


## Recovery vs Number of CWs and Strain


In [ ]:
if len(summary):
    group_cols = ["cfg_data_mode", "cfg_n_cw", "cfg_log10_h"]
    if "cfg_gwb_log10_a" in summary.columns:
        group_cols.append("cfg_gwb_log10_a")
    group_cols = [c for c in group_cols if c in summary.columns]
    g = summary.groupby(group_cols, dropna=False).agg(
        frac_half_mean=("frac_within_half_mode", "mean"),
        frac_half_max=("frac_within_half_mode", "max"),
        delta_lnL_best=("delta_lnL_truth", "max"),
        median_sigma=("median_abs_sigma", "median"),
        n=("file", "count"),
    ).reset_index()
    display(g)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for label, sub in g.groupby([c for c in ["cfg_data_mode", "cfg_log10_h"] if c in g.columns], dropna=False):
        sub = sub.sort_values("cfg_n_cw")
        axes[0].plot(sub["cfg_n_cw"], sub["frac_half_mean"], "o-", label=str(label))
        axes[1].plot(sub["cfg_n_cw"], sub["delta_lnL_best"], "o-", label=str(label))
        axes[2].plot(sub["cfg_n_cw"], sub["median_sigma"], "o-", label=str(label))
    axes[0].set_ylabel("mean fraction within 0.5 mode")
    axes[1].set_ylabel("best final lnL - truth lnL")
    axes[2].set_ylabel("median |distance error| / sigma")
    for ax in axes:
        ax.set_xlabel("N_CW")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    plt.tight_layout()
else:
    print("No summary rows.")


## Convergence Curves


In [ ]:
if len(history):
    fig, ax = plt.subplots(figsize=(8, 5))
    for (file, chain), sub in history.groupby(["file", "chain_id"]):
        sub = sub.sort_values("sweep")
        ax.plot(sub["sweep"], sub["frac_within_half_mode"], alpha=0.45)
    ax.set_xlabel("sweep")
    ax.set_ylabel("fraction within 0.5 mode")
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.3)
    plt.tight_layout()
else:
    print("No history rows.")


## Best Chain Per Run


In [ ]:
if len(summary):
    idx = summary.groupby("file")["final_lnL"].idxmax()
    best = summary.loc[idx].copy().sort_values(["cfg_n_cw", "cfg_log10_h", "cfg_n_psr"])
    display(best[["file", "cfg_n_cw", "cfg_log10_h", "cfg_n_psr", "final_lnL", "truth_lnL", "delta_lnL_truth", "frac_within_half_mode", "median_abs_modes", "max_abs_modes"]])

    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(best["median_abs_modes"], best["frac_within_half_mode"], c=best["cfg_n_cw"], s=70, cmap="viridis")
    cb = plt.colorbar(sc, ax=ax)
    cb.set_label("N_CW")
    ax.set_xscale("log")
    ax.set_xlabel("median |distance error| / min mode spacing")
    ax.set_ylabel("fraction within 0.5 mode")
    ax.grid(alpha=0.3)
    plt.tight_layout()


## Per-Pulsar Error Heatmap for One Run


In [ ]:
if files:
    RUN_FILE = files[-1]
    data = json.loads(RUN_FILE.read_text())
    names = [p["name"] for p in data["pulsars"]]
    truth = np.array([p["dist_mean_kpc"] for p in data["pulsars"]])
    dL = np.array([p["min_mode_spacing_kpc"] for p in data["pulsars"]])
    rec = np.array([r["recovered_distances"] for r in data["results"]])
    err_modes = (rec - truth[None, :]) / dL[None, :]

    fig, ax = plt.subplots(figsize=(max(10, len(names)*0.25), 3.5))
    im = ax.imshow(err_modes, aspect="auto", cmap="coolwarm", vmin=-2, vmax=2)
    ax.set_yticks(range(len(data["results"])))
    ax.set_yticklabels([f"chain {r['chain_id']} {r['init_kind']}" for r in data["results"]])
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    cb = plt.colorbar(im, ax=ax)
    cb.set_label("distance error / min mode spacing")
    ax.set_title(RUN_FILE.name)
    plt.tight_layout()
else:
    print("No files.")
